In [2]:
%pip install python-dotenv langchain-upstage


  Using cached pypdf-6.7.5-py3-none-any.whl.metadata (7.1 kB)
  Using cached tokenizers-0.20.3-cp313-none-win_amd64.whl.metadata (6.9 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached filelock-3.25.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
Using cached pypdf-6.7.5-py3-none-any.whl (331 kB)
Using cached tokenizers-0.20.3-cp313-none-win_amd64.whl (2.4 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached fsspec-2026.2.0-py3-none-any.whl (202 kB)
Using cached filelock-3.25.0-py3-none-any.whl (26 kB)

   ---------------------------------------- 0/6 [pypdf]
   ---------------------------------------- 0/6 [pypdf]
   ---------------------------------------- 0/6 [pypdf]
   ---------------------------------------- 0/6 [pypdf]
   ---------------------------------------- 0/6 [pypdf]
   ---------------------------------------- 0/6 [pypdf]
   --------------------------------

In [1]:
%pip install langchain langchain-community langchain-text-splitters langchain-pinecone docx2txt

  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_pinecone-0.2.13-py3-none-any.whl.metadata (8.6 kB)
  Using cached docx2txt-0.9-py3-none-any.whl.metadata (529 bytes)
  Using cached langgraph-1.0.10-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl.metadata (4.9 k

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [4]:
from langchain_upstage import ChatUpstage

llm = ChatUpstage()

ai_message = llm.invoke("한국에 대해 3줄로 설명해줘")

ai_message.content

'한국은 동북아시아에 위치한 국가로, 수도는 서울입니다. 한국은 고유의 문화와 역사를 가지고 있으며, K-pop, K-drama, K-movie 등 세계적인 인기를 끌고 있는 한류 문화의 발상지입니다. 또한, 한국은 IT 기술과 반도체 산업에서 세계적인 경쟁력을 가진 경제 대국 중 하나입니다.'

In [ ]:
# docx파일 load, split and saving in the vector storage

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./접구선통기술기준_20251128.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [6]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# 환경변수를 불러옴
load_dotenv()

# Upstage 제공하는 Embedding Model을 활용해서 `chunk`를 vector화
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 데이터를 처음 저장할 때 
index_name_gl = 'groundandlinelaw-index'
database1 = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name_gl)

In [8]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./방송통신설비기술기준_20240719.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
from langchain_pinecone import PineconeVectorStore

# 데이터를 처음 저장할 때 
index_name_bc = 'broadcastandcomlaw-index'
database2 = PineconeVectorStore.from_documents(document_list, embedding, index_name=index_name_bc)

In [ ]:
%pip install langchain-pinecone pinecone-client

In [ ]:
%pip install Pinecone

In [17]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

index_name_gl = 'groundandlinelaw-index'
index_name_bc = 'broadcastandcomlaw-index'

groundline_db = PineconeVectorStore(index_name=index_name_gl, embedding=embedding)
broadcom_db = PineconeVectorStore(index_name=index_name_bc, embedding=embedding)

In [18]:
query = '통신설비 접지 설치 시 접지저항 값은 얼마입니까?'

# `k` 값을 조절해서 얼마나 많은 데이터를 불러올지 결정
docs1 = groundline_db.similarity_search_with_score(query, k=3)
docs2 = broadcom_db.similarity_search_with_score(query, k=3)

all_docs = docs1 + docs2

all_docs = sorted(all_docs, key=lambda x:x[1],reverse=True)

In [19]:
all_docs

[(Document(id='aa79719e-d00f-4a61-81a5-4038a5ef2c5f', metadata={'source': './접구선통기술기준_20251128.docx'}, page_content='제4조(보호기 성능) ① 보호기의 기본회로도는 별표 1과 같으며, 보호기의 성능은 제2항 내지 제4항의 조건을 만족하여야 한다.\n\n② 보호기의 과전압 성능은 다음 각호와 같아야 한다.\n\n1. 보호기는 직류 100V/sec의 상승전압을 L1-E, L2-E간에 인가할 때 184V 이상 280V 이하에서 접지를 통하여 방전이 개시되어야 한다.\n\n2. 보호기는 100V/㎲의 상승전압을 L1-E, L2-E간에 인가할 때 180V 이상 600V 이하에서 접지를 통하여 방전되어야 한다.\n\n3. 보호기는 1000V/㎲의 상승전압을 L1-E, L2-E간에 인가할 때 180V 이상 700V 이하에서 접지를 통하여 방전되어야 한다.\n\n③ 보호기의 과전류 성능은 다음 각호와 같아야 한다.\n\n1. 보호기는 L1-T1, L2-T2간에 교류 110V 250㎃를 인가할 때 1분이내, 교류 110V 1A를 인가할 때 2초 이내에 동작하여 부동작 전류 이하로 전류를 제한하고, 과전류가 제거되면 자기 복구되어야 한다.\n\n2. 보호기는 L1-T1, L2-T2간에 직류 150㎃를 3시간 인가할 때 과전류 제한소자는 동작하지 않아야 한다.\n\n④ 보호기의 발화방지 성능은 다음 각호와 같아야 한다.\n\n1. 보호기는 L1-E, L2-E간에 60㎐, 5A를 15분간 인가할 때 과전압 방전소자의 발화방지 장치가 동작하여 보호기의 발화 및 변형이 없어야 한다.\n\n2. 보호기는 과전압 방전소자가 삽입되지 않은 상태에서 L1-T1, L2-T2간에 교류 220V, 3A을 15분간 인가할 때 과전류 제한소자가 손상되지 않아야 하며, 보호기의 발화 및 변형이 없어야 한다.\n\n\n\n제5조(접지저항 등) ① 교환설비ㆍ전송설비 및 통신케이블과 금속으로 된 단자함(구내통신단자함,